## Gradient-Weighted Class Activation Mapping (Grad-CAM)

In [ ]:
import torch
import numpy as np
import fabio
import yaml
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from maxima_vit.utils import load_model, image_to_tensor

MODEL_PATH = r""
IMAGE_PATH = r""
CONFIG_PATH = r""

In [ ]:
class GeometricParameterTarget:
    """
    Tells Grad-CAM which specific continuous variable in the 6D output 
    to backpropagate from.
    Indices: [0: dist, 1: poni1, 2: poni2, 3: rot1, 4: rot2, 5: rot3]
    """
    def __init__(self, param_index: int):
        self.param_index = param_index
        
    def __call__(self, model_output):
        if model_output.ndim == 1:
            return model_output[self.param_index]
        return model_output[0, self.param_index]

def swin_reshape_transform(tensor):
    """
    Transforms Swin intermediate outputs to (B, C, H, W) for Grad-CAM.
    Handles tuple/list outputs from block/layer hooks.
    """
    if isinstance(tensor, (tuple, list)):
        tensor = tensor[0]

    if tensor.ndim == 4:
        # Already image-like: (B, C, H, W) or (B, H, W, C)
        if tensor.shape[1] <= 4 and tensor.shape[-1] > 4:
            tensor = tensor.permute(0, 3, 1, 2)
        return tensor

    if tensor.ndim != 3:
        raise ValueError(f"Unexpected tensor shape for Grad-CAM reshape: {tuple(tensor.shape)}")

    batch_size, seq_len, channels = tensor.shape
    grid_size = int(np.sqrt(seq_len))

    if grid_size * grid_size == seq_len:
        spatial_tensor = tensor
    elif int(np.sqrt(seq_len - 1)) ** 2 == seq_len - 1:
        spatial_tensor = tensor[:, 1:, :]
        grid_size = int(np.sqrt(seq_len - 1))
    else:
        raise ValueError(f"Cannot infer spatial grid for sequence length {seq_len}.")

    result = spatial_tensor.reshape(batch_size, grid_size, grid_size, channels)
    result = result.permute(0, 3, 1, 2)
    return result

In [ ]:
def visualize_geometric_attention(model, input_tensor, original_image, 
                                  target_layer, param_index=0, param_name="Distance"):
    """
    Extracts and visualizes the attention map for a specific regression parameter.
    
    Args:
        model: Your loaded PyTorch model.
        input_tensor: The preprocessed image tensor (1, C, H, W).
        original_image: A numpy array of the image scaled [0, 1] for visualization.
        target_layer: The specific layer to extract gradients from (usually the last LayerNorm).
        param_index: The index of the 6D output to investigate (0 to 5).
        param_name: String label for the plot title.
    """
    model.eval()
    
    if original_image.ndim == 2:
        original_image = np.stack([original_image] * 3, axis=-1)
    elif original_image.ndim == 3 and original_image.shape[-1] == 1:
        original_image = np.repeat(original_image, 3, axis=-1)

    original_image = original_image.astype(np.float32)
    if np.max(original_image) > 1.0 or np.min(original_image) < 0.0:
        original_image = (original_image - np.min(original_image)) / (np.max(original_image) - np.min(original_image) + 1e-8)
    
    # initialize
    cam = GradCAM(model=model, 
                  target_layers=[target_layer], 
                  reshape_transform=swin_reshape_transform)
    
    targets = [GeometricParameterTarget(param_index)]
    
    # generate the heatmap
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets, eigen_smooth=True)[0, :]

    # match visualization image size to CAM map size
    cam_h, cam_w = grayscale_cam.shape
    if original_image.shape[0] != cam_h or original_image.shape[1] != cam_w:
        image_tensor = torch.from_numpy(original_image).permute(2, 0, 1).unsqueeze(0)
        image_tensor = torch.nn.functional.interpolate(
            image_tensor, size=(cam_h, cam_w), mode="bilinear", align_corners=False
        )
        original_image = image_tensor.squeeze(0).permute(1, 2, 0).numpy()

    visualization = show_cam_on_image(original_image, grayscale_cam, use_rgb=True)
    
    # plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    axes[0].imshow(original_image)
    axes[0].set_title("Original Diffraction Pattern", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(visualization)
    axes[1].set_title(f"Attention Map for: {param_name}", fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

model = load_model(MODEL_PATH, config)

In [ ]:
# Diagnostic: inspect the last encoder layer and its last block
layer = model.swin.encoder.layers[-1]
print("Layer type:", type(layer))
print("Layer children:", [name for name, _ in layer.named_children()])

# If the layer contains 'blocks', inspect the last block
if hasattr(layer, "blocks"):
    block = layer.blocks[-1]
    print("Block type:", type(block))
    print("Block children:", [name for name, _ in block.named_children()])

    # Print names and types for all child modules
    for name, mod in block.named_children():
        print(f"  - {name}: {type(mod)}")

    # Choose a candidate LayerNorm or attention module if present
    if hasattr(block, "norm2"):
        target_layer = block.norm2
    elif hasattr(block, "norm1"):
        target_layer = block.norm1
    elif hasattr(block, "attn"):
        target_layer = block.attn
    elif hasattr(block, "mlp"):
        target_layer = block.mlp
    else:
        # fallback: use the whole block (may produce non-tensor hook outputs)
        target_layer = block

    print("Selected target_layer:", type(target_layer))
else:
    print("No 'blocks' attribute on layer; printing layer repr:")
    print(layer)

In [ ]:

target_layer = model.swin.encoder.layers[-1].blocks[-1].output

input_image = fabio.open(IMAGE_PATH)
image_data = input_image.data.astype(np.float32)
input_tensor = image_to_tensor(image_data, config["model"].get("image_size", 224)).unsqueeze(0)

params = {
    0: "Distance",
    1: "Poni1",
    2: "Poni2",
    3: "Rot1",
    4: "Rot2",
    5: "Rot3"
}

for idx, name in params.items():
    visualize_geometric_attention(
        model=model,
        input_tensor=input_tensor,
        original_image=image_data / np.max(image_data),
        target_layer=target_layer,
        param_index=idx,
        param_name=name
    )